# Morphology by Condition, Over Every Shared Frame

Compares cell/nuclear morphology across all four conditions
(`naive`, `vem_1uM`, `gem_2.5nM`, `tram_6nM`) at every frame shared by
all of them (up to `config.MAX_FRAME`) - the four conditions don't
necessarily have the same number of frames, so `main.py` computes the
actual intersection rather than assuming one (see `select_cells.py`).
Every selected cell, whatever frame it's from, goes through the SAME
CAJAL run and the SAME UMAP/PHATE fit (`main.py` pools every condition
and every shared frame together before fitting), so they share one
embedding space - every frame's positions are directly comparable, not
independently-fit spaces that just happen to look similar. Frame is
simply a page within that one shared space, same as `naive_vs_vem`'s
own notebooks page through frames.

Requires `main.py` to have already been run (for `selected_metadata.csv`
and `{cell,nuc}_{umap,phate}.npz` under `output/cajal_output/`).


In [ ]:
import os
import sys

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/03_multi_drug_comparison/notebooks/), so the analysis dir -
# and config.py - is one level up, and the repo root - where shared/
# lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/MASKS_DIR/CAJAL_OUTPUT_DIR are relative to the
# analysis directory, not this notebook's own cwd - re-anchor them to an
# absolute path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.MASKS_DIR        = os.path.join(ANALYSIS_DIR, config.MASKS_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.load_data import load_full_dataset, export_plot_data
from shared.density_plots import export_kde_by_frame


## Method

For each embedding method (UMAP, PHATE), each modality (cell, nucleus):

1. **Load the shared embedding** (`load_full_dataset`): `main.py` runs
   CAJAL/UMAP/PHATE once across every condition and both frames
   together, so this is already exactly the right data, no subsetting
   needed.
2. **Plot**: one combined PDF (one page per frame, with fixed shared
   axis limits across every page so they're directly comparable) and
   one SVG per frame, with a KDE contour per condition
   (`export_kde_by_frame`).


In [ ]:
CONDITION_COLORS = {
    'naive':     'tab:blue',
    'vem_1uM':   'tab:red',
    'gem_2.5nM': 'tab:orange',
    'tram_6nM':  'tab:green',
}
NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '01_kde_by_condition')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# KDE contour levels (level p encloses the densest (1-p) fraction of a
# group's mass - higher is tighter/closer to the peak) and bandwidth
# multiplier (above 1 smooths over sampling/distance-tie noise)
KDE_LEVELS     = (0.7, 0.8, 0.9)
KDE_BW_ADJUST  = 1

# main.py computes both of these - plot both every time this notebook
# runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_modality(modality):
    '''
    For each embedding method, load this modality's shared embedding
    (every condition, every shared frame) and export a PDF + per-frame
    SVG, with a KDE contour per condition.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        dfs: {embedding_method: the full merged dataset for this
              modality and embedding method} (in-memory only)
    '''
    dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        print(f"[{modality}/{embedding_method}] {len(df)} cells across {df['frame'].nunique()} frames, "
              f"{df['condition'].nunique()} conditions")

        export_kde_by_frame(df, 'condition', CONDITION_COLORS, NOTEBOOK_PLOTS_DIR,
                             f'{modality}_{embedding_method}', embedding_method.upper(),
                             title_prefix=f'{MODALITY_NAMES[modality]} morphology',
                             levels=KDE_LEVELS, bw_adjust=KDE_BW_ADJUST)
        dfs[embedding_method] = df

    return dfs


## Run: cell morphology

In [ ]:
cell_dfs = process_modality('cell')


## Run: nuclear morphology

In [ ]:
nuc_dfs = process_modality('nuc')


## Export plot data

Combines every (modality, embedding method) DataFrame above into one CSV
(`plot_data.csv`, alongside the PDFs/SVGs above) - the same dim_0/dim_1
coordinates and metadata the KDE plots above use, so someone without
this repo can recreate or extend these plots from the raw data alone.

In [ ]:
plot_data = {('cell', m): df for m, df in cell_dfs.items()}
plot_data.update({('nuc', m): df for m, df in nuc_dfs.items()})
export_plot_data(plot_data, os.path.join(NOTEBOOK_PLOTS_DIR, 'plot_data.csv'))
